In [11]:
import os
import csv
import datetime as dt
import pandas as pd

# index를 날짜로 하는 데이터프레임을 price, divend에 대해 반환
def extract_etf_data(file_path, etf_name):
    extract_datas = []

    with open(os.path.join(file_path, etf_name + '.csv'), newline='') as csvfile:
        reader = csv.reader(csvfile)

        for row in reader:
            extract_datas.append(row)
        
    price_dates = list(map(dt.date.fromisoformat, extract_datas[0]))
    price_history = list(map(float, extract_datas[1]))
    divend_dates = list(map(dt.date.fromisoformat, extract_datas[2]))
    divend_history = list(map(float, extract_datas[3]))

    price_df = pd.DataFrame({'date': price_dates, 'price': price_history})
    divend_df = pd.DataFrame({'date': divend_dates, 'divend': divend_history})

    price_df.set_index('date', inplace=True)
    divend_df.set_index('date', inplace=True)

    return (price_df, divend_df)


In [17]:
def make_etf_data(file_path, etf_names):
    prices = []
    divends = []

    for etf in etf_names:
        price_df, divend_df = extract_etf_data(file_path, etf)
        price_df.rename(columns={'price': etf}, inplace=True)
        divend_df.rename(columns={'divend': etf}, inplace=True)
        prices.append(price_df)
        divends.append(divend_df)

    etf_price_df = pd.concat(prices, axis=1, sort=True)
    etf_divend_df = pd.concat(divends, axis=1, sort=True)

    return {'price': etf_price_df, 'divend': etf_divend_df}

In [ ]:
class Portfolio:
    # etf_data
    def __init__(self, etf_data: dict, start_cash: float, target_ratio: dict, buy_ratio: float = 5.0, sell_ratio: float = 5.0):
        self.__price_data = etf_data['price'].copy()
        self.__divend_data = etf_data['divend'].copy()
        self.__start_cash = start_cash
        # target_ratio의 구성 { etf명 : 목표 비중, ... }
        self.target_ratio = target_ratio
        self.buy_ratio = buy_ratio
        self.sell_ratio = sell_ratio

        self.__price_data.dropna(inplace=True)
        self.__avilable_date = self.price_data.iloc[0].name
        self.start_date = self.__avilable_date
        self.__divend_data = self.__divend_data[self.__divend_data.index >= self.start_date]
    @property
    def price_data(self):
        return self.__price_data
    @property
    def divend_data(self):
        return self.__divend_data

    def backtest(self):
        col = ['total_value', 'cash', ]

        stat = pd.DataFrame(columns=['cash', 'total_value', 'profit'])
        stat.index.name = 'date'

In [41]:
# etf 불러와서 데이터프레임화 하기
file_path = './ETF'
etf_names = ['QQQ', 'DGRW', 'SCHD']

etf_data = make_etf_data(file_path, etf_names)

In [48]:
Portfolio(etf_data, 1000000, {'QQQ': 0.5, 'DGRW': 0.3, 'SCHD': 0.2}, buy_ratio=5.0, sell_ratio=5.0)

              QQQ   DGRW   SCHD
date                           
2003-12-24  0.014    NaN    NaN
2004-12-17  0.379    NaN    NaN
2005-06-17  0.035    NaN    NaN
2005-12-16  0.101    NaN    NaN
2006-03-17  0.029    NaN    NaN
...           ...    ...    ...
2026-02-24    NaN  0.055    NaN
2026-03-23  0.733    NaN    NaN
2026-03-25    NaN    NaN  0.257
2026-03-26    NaN  0.170    NaN
2026-04-27    NaN  0.075    NaN

[281 rows x 3 columns]
              QQQ   DGRW   SCHD
date                           
2013-06-21  0.224    NaN    NaN
2013-06-24    NaN  0.038  0.075
2013-07-22    NaN  0.038    NaN
2013-08-26    NaN  0.038    NaN
2013-09-20  0.238    NaN    NaN
...           ...    ...    ...
2026-02-24    NaN  0.055    NaN
2026-03-23  0.733    NaN    NaN
2026-03-25    NaN    NaN  0.257
2026-03-26    NaN  0.170    NaN
2026-04-27    NaN  0.075    NaN

[240 rows x 3 columns]
